In [2]:
# ── Recovery: Drive, paths, data, splits ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, glob, cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers

PROJECT_DIR = '/content/drive/MyDrive/skin_cancer_project'
DATASET_DIR = f'{PROJECT_DIR}/ham10000'
MODEL_PATH  = f'{PROJECT_DIR}/best_model.keras'          # your original EfficientNetB0
RESNET_PATH = f'{PROJECT_DIR}/best_model_resnet50.keras'  # new — ResNet-50 will save here
IMG_SIZE    = 224
BATCH_SIZE  = 32
SEED        = 42

CLASS_NAMES = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
CLASS_DISPLAY = {
    'nv': 'Melanocytic nevi', 'mel': 'Melanoma', 'bkl': 'Benign keratosis',
    'bcc': 'Basal cell carcinoma', 'akiec': 'Actinic keratosis',
    'vasc': 'Vascular lesion', 'df': 'Dermatofibroma'
}
CLASS_LABELS = {name: i for i, name in enumerate(CLASS_NAMES)}

# Metadata + paths
metadata = pd.read_csv(f'{DATASET_DIR}/HAM10000_metadata.csv')
path_lookup = {}
for folder in glob.glob(f'{DATASET_DIR}/HAM10000_images*'):
    for fp in glob.glob(f'{folder}/*.jpg'):
        path_lookup[os.path.splitext(os.path.basename(fp))[0]] = fp
metadata['path']  = metadata['image_id'].map(path_lookup)
metadata['label'] = metadata['dx'].map(CLASS_LABELS)
metadata = metadata.dropna(subset=['path']).reset_index(drop=True)

# EXACT same split as your original run
train_val, test_df = train_test_split(
    metadata, test_size=0.15, random_state=SEED, stratify=metadata['label']
)
train_df, val_df = train_test_split(
    train_val, test_size=0.18, random_state=SEED, stratify=train_val['label']
)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

cw_array = compute_class_weight(
    class_weight='balanced', classes=np.arange(len(CLASS_NAMES)), y=train_df['label'].values
)
class_weights = {i: w for i, w in enumerate(cw_array)}

# EXACT same tf.data pipeline — pixels stay 0–255, model normalizes internally
def load_image(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, max_delta=0.1)
        img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def make_dataset(df, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df['path'].values, df['label'].values.astype(np.int32))
    )
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED)
    ds = ds.map(lambda p, l: load_image(p, l, augment), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, augment=True,  shuffle=True)
val_ds   = make_dataset(val_df,   augment=False, shuffle=False)
test_ds  = make_dataset(test_df,  augment=False, shuffle=False)

print('✅ Pipeline rebuilt — identical to original training run')

Mounted at /content/drive
Train: 6979 | Val: 1533 | Test: 1503
✅ Pipeline rebuilt — identical to original training run


In [3]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = resnet_preprocess(inputs)             # ResNet's own normalization — NOT EfficientNet's
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(7, activation='softmax')(x)
resnet_model = tf.keras.Model(inputs, outputs)

total = resnet_model.count_params()
trainable = sum(tf.size(w).numpy() for w in resnet_model.trainable_weights)
print(f'Total params: {total:,}')
print(f'Trainable (Phase A): {trainable:,}')
print(f'(EfficientNetB0 was ~4.01M total)')

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Total params: 23,602,055
Trainable (Phase A): 14,343
(EfficientNetB0 was ~4.01M total)


In [4]:
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_a_resnet = resnet_model.fit(
    train_ds, validation_data=val_ds, epochs=20,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(RESNET_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1)
    ],
    verbose=1
)
print('✅ Phase A done. Best val accuracy:', f"{max(history_a_resnet.history['val_accuracy']):.1%}")

Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.1309 - loss: 2.7394
Epoch 1: val_accuracy improved from None to 0.36986, saving model to /content/drive/MyDrive/skin_cancer_project/best_model_resnet50.keras

Epoch 1: finished saving model to /content/drive/MyDrive/skin_cancer_project/best_model_resnet50.keras
219/219 ━━━━━━━━━━━━━━━━━━━━ 676s 3s/step - accuracy: 0.1952 - loss: 2.5436 - val_accuracy: 0.3699 - val_loss: 1.6484
Epoch 2/20
218/219 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step - accuracy: 0.3165 - loss: 2.0643
Epoch 2: val_accuracy improved from 0.36986 to 0.54729, saving model to /content/drive/MyDrive/skin_cancer_project/best_model_resnet50.keras

Epoch 2: finished saving model to /content/drive/MyDrive/skin_cancer_project/best_model_resnet50.keras
219/219 ━━━━━━━━━━━━━━━━━━━━ 68s 311ms/step - accuracy: 0.3533 - loss: 1.9272 - val_accuracy: 0.5473 - val_loss: 1.3249
Epoch 3/20
218/219 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step - accuracy: 0.4260 - loss: 1.7627
Epoch 3: val_a

In [5]:
base_model.trainable = True
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_b_resnet = resnet_model.fit(
    train_ds, validation_data=val_ds, epochs=20,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(RESNET_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1)
    ],
    verbose=1
)
print('✅ Phase B done. Best val accuracy:', f"{max(history_b_resnet.history['val_accuracy']):.1%}")

Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 460ms/step - accuracy: 0.5853 - loss: 1.4670
Epoch 1: val_accuracy improved from None to 0.68558, saving model to /content/drive/MyDrive/skin_cancer_project/best_model_resnet50.keras

Epoch 1: finished saving model to /content/drive/MyDrive/skin_cancer_project/best_model_resnet50.keras
219/219 ━━━━━━━━━━━━━━━━━━━━ 206s 576ms/step - accuracy: 0.5923 - loss: 1.2525 - val_accuracy: 0.6856 - val_loss: 0.8670
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step - accuracy: 0.6325 - loss: 0.8570
Epoch 2: val_accuracy improved from 0.68558 to 0.71233, saving model to /content/drive/MyDrive/skin_cancer_project/best_model_resnet50.keras

Epoch 2: finished saving model to /content/drive/MyDrive/skin_cancer_project/best_model_resnet50.keras
219/219 ━━━━━━━━━━━━━━━━━━━━ 130s 409ms/step - accuracy: 0.6381 - loss: 0.8434 - val_accuracy: 0.7123 - val_loss: 0.7860
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 368ms/step - accuracy: 0.6825 - loss: 0.6625
Epoch 3

In [6]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
import json

resnet_model = tf.keras.models.load_model(RESNET_PATH)

y_true, y_pred_probs = [], []
for imgs, lbls in test_ds:
    y_pred_probs.extend(resnet_model.predict(imgs, verbose=0))
    y_true.extend(lbls.numpy())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = np.mean(y_pred == y_true)
y_bin = label_binarize(y_true, classes=list(range(7)))
auc = roc_auc_score(y_bin, y_pred_probs, multi_class='ovr', average='macro')

print(f'ResNet-50 Test Accuracy : {accuracy:.1%}')
print(f'ResNet-50 AUC-ROC (macro): {auc:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=[CLASS_DISPLAY[c] for c in CLASS_NAMES]))

cm = confusion_matrix(y_true, y_pred)
print('Confusion matrix:')
print(cm)

# Save for the paper
results = {
    'model': 'ResNet-50',
    'total_params': int(total),
    'test_accuracy': float(accuracy),
    'auc_roc_macro': float(auc),
    'confusion_matrix': cm.tolist(),
}
with open(f'{PROJECT_DIR}/resnet50_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('✅ Saved resnet50_results.json to Drive')

ResNet-50 Test Accuracy : 81.6%
ResNet-50 AUC-ROC (macro): 0.9633

                      precision    recall  f1-score   support

    Melanocytic nevi       0.94      0.88      0.91      1006
            Melanoma       0.52      0.62      0.57       167
    Benign keratosis       0.64      0.68      0.66       165
Basal cell carcinoma       0.76      0.77      0.76        77
   Actinic keratosis       0.58      0.69      0.63        49
     Vascular lesion       0.83      0.91      0.87        22
      Dermatofibroma       0.60      0.88      0.71        17

            accuracy                           0.82      1503
           macro avg       0.69      0.78      0.73      1503
        weighted avg       0.83      0.82      0.82      1503

Confusion matrix:
[[883  70  35   8   6   1   3]
 [ 34 104  17   3   8   0   1]
 [ 16  22 112   5   8   0   2]
 [  5   2   2  59   3   3   3]
 [  1   2   9   2  34   0   1]
 [  2   0   0   0   0  20   0]
 [  1   0   0   1   0   0  15]]
✅ Saved resn

In [8]:
model = tf.keras.models.load_model(MODEL_PATH)
print('✅ EfficientNetB0 loaded from Drive')

# Kaggle setup (skip if already done earlier in this session)
from google.colab import files
import os
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json...')
    files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

PH2_DIR = f'{PROJECT_DIR}/ph2'
os.makedirs(PH2_DIR, exist_ok=True)
os.system(f'kaggle datasets download -d spacesurfer/ph2-dataset -p {PH2_DIR}')
os.system(f'unzip -q {PH2_DIR}/ph2-dataset.zip -d {PH2_DIR}')
print('✅ PH2 downloaded')

✅ EfficientNetB0 loaded from Drive
Upload your kaggle.json...


Saving kaggle.json to kaggle.json
✅ PH2 downloaded


In [9]:
# PH2 mirrors on Kaggle vary in layout — let's see exactly what we got
import subprocess
print(subprocess.run(['find', PH2_DIR, '-maxdepth', '3'], capture_output=True, text=True).stdout[:3000])

/content/drive/MyDrive/skin_cancer_project/ph2
/content/drive/MyDrive/skin_cancer_project/ph2/ph2-dataset.zip
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD002
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD003
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD004
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD006
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD008
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD009
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD010
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD013
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD014

In [10]:
import subprocess

# 1. What's inside one lesion folder?
print("=== Inside IMD002 ===")
print(subprocess.run(['find', f'{PH2_DIR}/PH2Dataset/PH2 Dataset images/IMD002'],
                      capture_output=True, text=True).stdout)

# 2. Find the metadata/labels file (diagnosis info) anywhere in the PH2 folder
print("\n=== Searching for metadata/labels file ===")
print(subprocess.run(['find', f'{PH2_DIR}/PH2Dataset', '-maxdepth', '2',
                       '-iname', '*.txt', '-o', '-iname', '*.xlsx', '-o', '-iname', '*.xls', '-o', '-iname', '*.csv'],
                      capture_output=True, text=True).stdout)

# 3. Just in case, full top-level listing of PH2Dataset (not images subfolder)
print("\n=== Top level of PH2Dataset ===")
print(subprocess.run(['ls', '-la', f'{PH2_DIR}/PH2Dataset'],
                      capture_output=True, text=True).stdout)

=== Inside IMD002 ===
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD002
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD002/IMD002_Dermoscopic_Image
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD002/IMD002_Dermoscopic_Image/IMD002.bmp
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD002/IMD002_lesion
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD002/IMD002_lesion/IMD002_lesion.bmp
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD002/IMD002_roi
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD002/IMD002_roi/IMD002_R1_Label4.bmp
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2 Dataset images/IMD002/IMD002_roi/IMD002_R2_Label3.bmp


=== Searching for metadata/labels file ===
/content/drive/MyDrive/skin_cancer_project/ph2/PH2Dataset/PH2_dataset.txt
/cont

In [11]:
import pandas as pd

# Peek at the Excel label file (usually cleaner than the .txt)
xlsx_path = f'{PH2_DIR}/PH2Dataset/PH2_dataset.xlsx'
ph2_raw = pd.read_excel(xlsx_path, header=None)  # header=None first, since PH2 files often have a multi-row preamble

print("Shape:", ph2_raw.shape)
print("\nFirst 15 rows, all columns:")
print(ph2_raw.head(15).to_string())

Shape: (213, 17)

First 15 rows, all columns:
            0                       1                   2               3         4                                                                             5                        6                        7               8                        9                         10      11   12           13          14         15     16
0          NaN                     NaN                 NaN             NaN       NaN                                                                           NaN                      NaN                      NaN             NaN                      NaN                       NaN     NaN  NaN          NaN         NaN        NaN    NaN
1          NaN                     NaN                 NaN             NaN       NaN                                                                       Legends                      NaN                      NaN             NaN                      NaN                       NaN   

In [12]:
import pandas as pd
import os

xlsx_path = f'{PH2_DIR}/PH2Dataset/PH2_dataset.xlsx'
ph2_raw = pd.read_excel(xlsx_path, header=None)

# Real data rows: 13 onward (row 12 is the header row, rows 0-11 are legend/title)
ph2_data = ph2_raw.iloc[13:, [0, 2, 3, 4]].copy()
ph2_data.columns = ['image_name', 'common_nevus', 'atypical_nevus', 'melanoma']
ph2_data = ph2_data.dropna(subset=['image_name']).reset_index(drop=True)

print(f"Total PH2 entries parsed: {len(ph2_data)}")  # should be 200

# Binary label: melanoma=1 (col marked 'X'), else 0 (common or atypical nevus)
ph2_data['binary_label'] = ph2_data['melanoma'].notna().astype(int)
print(f"\nClass counts:")
print(f"  Nevus (common+atypical): {(ph2_data['binary_label']==0).sum()}")
print(f"  Melanoma: {(ph2_data['binary_label']==1).sum()}")
# Expect: 160 nevus, 40 melanoma

# Build full image paths
PH2_IMG_BASE = f'{PH2_DIR}/PH2Dataset/PH2 Dataset images'
def ph2_image_path(name):
    return f'{PH2_IMG_BASE}/{name}/{name}_Dermoscopic_Image/{name}.bmp'

ph2_data['image_path'] = ph2_data['image_name'].apply(ph2_image_path)

# Verify all images exist
ph2_data['exists'] = ph2_data['image_path'].apply(os.path.exists)
print(f"\nImages found: {ph2_data['exists'].sum()} / {len(ph2_data)}")
if ph2_data['exists'].sum() < len(ph2_data):
    print("Missing examples:")
    print(ph2_data[~ph2_data['exists']][['image_name', 'image_path']].head())

ph2_data = ph2_data[ph2_data['exists']].reset_index(drop=True)
print(f"\n✅ Final usable PH2 set: {len(ph2_data)} images")
print(ph2_data[['image_name', 'binary_label']].head())

Total PH2 entries parsed: 200

Class counts:
  Nevus (common+atypical): 160
  Melanoma: 40

Images found: 200 / 200

✅ Final usable PH2 set: 200 images
  image_name  binary_label
0     IMD003             0
1     IMD009             0
2     IMD016             0
3     IMD022             0
4     IMD024             0


In [13]:
import cv2
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import json

# `model` should already be loaded from Cell 5 (your trained EfficientNetB0)

def load_ph2_image(path):
    img = cv2.imread(path)  # cv2 handles .bmp natively
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype('float32')  # keep 0–255, same as your pipeline — model normalizes internally
    return img

images = np.array([load_ph2_image(p) for p in ph2_data['image_path']])
print(f"Loaded {len(images)} PH2 images, shape: {images.shape}")

# Batch inference
probs = model.predict(images, batch_size=BATCH_SIZE, verbose=1)  # shape (200, 7)

# Restrict to mel vs nv (your model's 7-class output, indices from CLASS_NAMES)
mel_idx = CLASS_NAMES.index('mel')
nv_idx  = CLASS_NAMES.index('nv')

p_mel = probs[:, mel_idx]
p_nv  = probs[:, nv_idx]
p_mel_binary = p_mel / (p_mel + p_nv + 1e-8)   # renormalize to a binary decision
y_pred_binary = (p_mel_binary > 0.5).astype(int)
y_true_binary = ph2_data['binary_label'].values

# Evaluate
acc = accuracy_score(y_true_binary, y_pred_binary)
auc = roc_auc_score(y_true_binary, p_mel_binary)
cm = confusion_matrix(y_true_binary, y_pred_binary)

print(f"\n{'='*55}")
print(f"PH2 Cross-Dataset Validation (binary mel vs. nevus)")
print(f"{'='*55}")
print(f"n = {len(y_true_binary)}")
print(f"Accuracy: {acc:.4f} ({acc*100:.1f}%)")
print(f"AUC-ROC: {auc:.4f}")
print(f"\nConfusion matrix [[TN, FP], [FN, TP]]:")
print(cm)
print(f"\n{classification_report(y_true_binary, y_pred_binary, target_names=['nevus','melanoma'], digits=2)}")

# Save for the paper
ph2_results = {
    'dataset': 'PH2 (external)',
    'task': 'binary mel-vs-nv',
    'n_images': int(len(y_true_binary)),
    'accuracy': float(acc),
    'auc_roc': float(auc),
    'confusion_matrix': cm.tolist(),
}
with open(f'{PROJECT_DIR}/ph2_results.json', 'w') as f:
    json.dump(ph2_results, f, indent=2)
print('\n✅ Saved ph2_results.json to Drive')

Loaded 200 PH2 images, shape: (200, 224, 224, 3)
7/7 ━━━━━━━━━━━━━━━━━━━━ 33s 2s/step

PH2 Cross-Dataset Validation (binary mel vs. nevus)
n = 200
Accuracy: 0.7150 (71.5%)
AUC-ROC: 0.8239

Confusion matrix [[TN, FP], [FN, TP]]:
[[112  48]
 [  9  31]]

              precision    recall  f1-score   support

       nevus       0.93      0.70      0.80       160
    melanoma       0.39      0.78      0.52        40

    accuracy                           0.71       200
   macro avg       0.66      0.74      0.66       200
weighted avg       0.82      0.71      0.74       200


✅ Saved ph2_results.json to Drive
